## Imports

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as f
from torchvision import datasets, transforms
import torchvision.datasets.oxford_iiit_pet as Data
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device:{device}')

Using device:cpu


## Load Dataset

In [9]:
IMG_SIZE = 256
BATCH_SIZE = 8

img_transform = transforms.Compose([
 transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
 transforms.ToTensor(),
 transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

mask_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor()
])

train_set = datasets.OxfordIIITPet(
    root='data',
    split='trainval',
    target_types='segmentation',
    download=True,
    transform=img_transform,
    target_transform=mask_transform,
)

test_set = datasets.OxfordIIITPet(
    root='data',
    split='test',
    target_types='segmentation',
    transform=img_transform,
    target_transform=mask_transform,
    download=True
)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'Train samples: {len(train_set)} | Test samples: {len(test_set)}')


Train samples: 3680 | Test samples: 3669


## Encoder

In [6]:


class Encoder(nn.Module):
  def __init__(self, in_channels, num_filters):
    super().__init__()
    self.block = nn.Sequential(
        nn.Conv2d(in_channels, num_filters, kernel_size=3),
        nn.ReLU(),
        nn.Conv2d(in_channels, num_filters, kernel_size=3),
        nn.ReLU()
    )

    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

  def forward(self, x):
      features = self.block(x)
      pooled = self.pool(features)
      return pooled, features


## Decoder

In [7]:
class Decoder(nn.Module):
    def __init__(self, in_channels, num_filters):
      super().__init__()

      self.upsample = nn.ConvTranspose2d(in_channels, num_filters, kernel_size=2, stride=2)

      #QUESTION: When we half the number of out channels during convolution, the kernel has no involvement in the process of halving the channel,
      #is that correct?
      self.conv_block = nn.Sequential(
          nn.Conv2d(in_channels*2, num_filters, kernel_size=3),
          nn.ReLU(),
          nn.Conv2d(in_channels, num_filters, kernel_size=3),
          nn.ReLU(),
      )

    def forward(self, x, skip_features):
        x = self.upsample(x)
        skip = f.interpolate(skip_features, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)  #?
        x = torch.cat([x, skip], dim=1)
        return self.conv_block(x)



## The U-Net model

In [8]:
class UNet(nn.Module):
  def __init__(self, in_channels=3, num_classes=1):
    super().__init__()

    # Contracting part
    self.enc1 = Encoder(in_channels, 64)
    self.enc2 = Encoder(64, 128)
    self.enc3 = Encoder(128, 256)
    self.enc4 = Encoder(256, 512)

    # Bottleneck
    self.bottleneck = nn.Sequential(
        nn.Conv2d(512, 1024, kernel_size=3),
        nn.ReLU,
        nn.Conv2d(512, 1024, kernel_size=3),
        nn.ReLU,
    )

    # Decoder
    self.dec1 = Decoder(1024, 512)
    self.dec2 = Decoder(512, 256)
    self.dec3 = Decoder(256, 128)
    self.dec4 = Decoder(128, 64)

    # Final 1x1 conv
    self.final = nn.Conv2d(64, num_classes, kernel_size=1)

  def forward(self, x):
    # Encoder: colelct the skip connections
    x, s1 = self.enc1(x)
    x, s2 = self.enc2(x)
    x, s3 = self.enc3(x)
    x, s4 = self.enc4(x)

    x = self.bottleneck(x)

    x = self.dec1(x, s4)
    x = self.dec1(x, s3)
    x = self.dec1(x, s2)
    x = self.dec1(x, s1)

    return self.final(x)

## Training and Evaluation Helpers


## Train

## Training Curve

## Visualize Predictions